In [9]:
import pandas as pd
import re

# Cargar archivo original
url_clientes = "https://raw.githubusercontent.com/kmeza3278/etl-data-pipeline-17-3278-2021-/refs/heads/main/data/F_clientes.csv"
clientes = pd.read_csv(url_clientes)

# Normalizar columnas
clientes.columns = clientes.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)

# Limpieza base
for col in clientes.select_dtypes(include="object").columns:
    clientes[col] = clientes[col].astype(str).str.strip()
    clientes[col] = clientes[col].replace(["nan", "None", "NULL", "null", ""], pd.NA)

# Limpiar correo
def limpiar_email(valor):
    if pd.isna(valor):
        return pd.NA
    valor = str(valor).strip().lower()
    valor = re.sub(r"\s+", "", valor)
    return valor if valor else pd.NA

# Validar correo
def email_valido(email):
    if pd.isna(email):
        return False
    patron = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
    return bool(re.match(patron, str(email)))

# Limpiar teléfono
def limpiar_telefono(valor):
    if pd.isna(valor):
        return pd.NA
    solo_digitos = re.sub(r"\D", "", str(valor))
    if len(solo_digitos) > 8 and solo_digitos.startswith("503"):
        solo_digitos = solo_digitos[-8:]
    return solo_digitos if solo_digitos else pd.NA

def telefono_valido(telefono):
    if pd.isna(telefono):
        return False
    return bool(re.fullmatch(r"\d{8}", str(telefono)))

# Limpiar nombre del cliente
def limpiar_nombre(nombre):
    if pd.isna(nombre):
        return pd.NA
    nombre = str(nombre).strip()

    # quitar números al final
    nombre = re.sub(r"\s+\d+\s*$", "", nombre)

    # quitar espacios duplicados
    nombre = re.sub(r"\s+", " ", nombre).strip()

    return nombre if nombre else pd.NA

clientes["cliente"] = clientes["cliente"].apply(limpiar_nombre)
clientes["correo"] = clientes["correo"].apply(limpiar_email)
clientes["telefono"] = clientes["telefono"].apply(limpiar_telefono)

# Motivo de rechazo
def motivo_rechazo(row):
    motivos = []

    if pd.isna(row["id_cliente"]):
        motivos.append("id_cliente_vacio")

    if pd.isna(row["cliente"]):
        motivos.append("cliente_vacio")

    if pd.isna(row["correo"]):
        motivos.append("correo_vacio")
    elif not email_valido(row["correo"]):
        motivos.append("correo_invalido")

    if pd.isna(row["telefono"]):
        motivos.append("telefono_vacio")
    elif not telefono_valido(row["telefono"]):
        motivos.append("telefono_invalido")

    return ",".join(motivos)

# Separar curated y rejects
clientes_curated = clientes[
    clientes["id_cliente"].notna() &
    clientes["cliente"].notna() &
    clientes["correo"].notna() &
    clientes["telefono"].notna() &
    clientes["correo"].apply(email_valido) &
    clientes["telefono"].apply(telefono_valido)
].copy()

clientes_rejects = clientes[
    ~(
        clientes["id_cliente"].notna() &
        clientes["cliente"].notna() &
        clientes["correo"].notna() &
        clientes["telefono"].notna() &
        clientes["correo"].apply(email_valido) &
        clientes["telefono"].apply(telefono_valido)
    )
].copy()

clientes_rejects["motivo_rechazo"] = clientes_rejects.apply(motivo_rechazo, axis=1)

# Exportar
clientes_curated.to_csv("clientes_curated.csv", index=False)
clientes_rejects.to_csv("clientes_rejects.csv", index=False)

print("Curated:", len(clientes_curated))
print("Rejects:", len(clientes_rejects))
print(clientes_curated.head(10))

from google.colab import files

files.download("clientes_curated.csv")
files.download("clientes_rejects.csv")

Curated: 125
Rejects: 20
   id_cliente           cliente                  correo  telefono
0      CL1000     Paola Mendoza  cliente061@empresa.com  73372703
1      CL1001     Elena Ramírez  cliente186@outlook.com  75447071
2      CL1002  Valeria Martínez   cliente25@outlook.com  76605966
3      CL1003    Daniela Rivera    cliente317@correo.sv  73134898
4      CL1004     Elena Morales    cliente426@correo.sv  78399637
7      CL1007   Marta Hernández  cliente735@empresa.com  72952127
8      CL1008      Diego Torres  cliente870@empresa.com  79305719
9      CL1009   Daniela Morales    cliente987@gmail.com  79050212
11     CL1011    Daniela Santos    cliente117@gmail.com  77253653
13     CL1013     Carlos Flores   cliente1367@gmail.com  74871926


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>